# Phase 3: Pretrained Baseline Evaluator Diagnostic (Hindi)
This notebook runs exactly 20 test examples through the real pretrained Phase 2 checkpoint to diagnose extraction and EOS behavior before any fine-tuning.

In [ ]:
import os

CHECKPOINT_PATH = "/kaggle/input/notebooks/negative00/notebookb7b29f6528/phase2_outputs/hindi_checkpoints/step_00020000.pt"
TOKENIZER_PATH  = "/kaggle/input/datasets/negative00/phase2-cloud-data/hindi/tokenizer/models/hindi_bpe_10k_final.model"
TEST_PATH       = "/kaggle/input/datasets/negative00/phase-3-finetune-dataset/hindi_test.jsonl"
SEED            = 42
NUM_EXAMPLES    = 20
MAX_NEW_TOKENS  = 20

print("Configuration loaded.")

In [ ]:
import math, torch, torch.nn as nn

class LMConfig:
    vocab_size: int = 10000
    d_model: int = 448
    n_layers: int = 8
    n_heads: int = 7
    d_ff: int = 2048
    max_seq_len: int = 512
    dropout: float = 0.1
    bias: bool = True

class InputEmbeddings(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.max_seq_len = config.max_seq_len
        self.token_embedding    = nn.Embedding(config.vocab_size, config.d_model)
        self.position_embedding = nn.Embedding(config.max_seq_len, config.d_model)
        self.dropout = nn.Dropout(config.dropout)
    def forward(self, input_ids):
        B, T = input_ids.shape
        pos = torch.arange(T, dtype=torch.long, device=input_ids.device)
        return self.dropout(self.token_embedding(input_ids) + self.position_embedding(pos))

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.n_heads  = config.n_heads
        self.head_dim = config.d_model // config.n_heads
        self.max_seq_len = config.max_seq_len
        self.q_proj   = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.k_proj   = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.v_proj   = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.out_proj = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.attn_dropout  = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        mask = torch.tril(torch.ones(config.max_seq_len, config.max_seq_len)).view(1,1,config.max_seq_len,config.max_seq_len)
        self.register_buffer("bias_mask", mask)
    def forward(self, x, return_attention=False):
        B, T, C = x.size()
        q = self.q_proj(x).view(B,T,self.n_heads,self.head_dim).transpose(1,2)
        k = self.k_proj(x).view(B,T,self.n_heads,self.head_dim).transpose(1,2)
        v = self.v_proj(x).view(B,T,self.n_heads,self.head_dim).transpose(1,2)
        att = (q @ k.transpose(-2,-1)) * (1.0/math.sqrt(self.head_dim))
        att = att.masked_fill(self.bias_mask[:,:,:T,:T]==0, float("-inf"))
        att = torch.nn.functional.softmax(att, dim=-1)
        y   = self.attn_dropout(att) @ v
        y   = y.transpose(1,2).contiguous().view(B,T,C)
        y   = self.resid_dropout(self.out_proj(y))
        if return_attention: return y, att
        return y

class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.d_model, config.d_ff,    bias=config.bias), nn.GELU(), nn.Dropout(config.dropout),
            nn.Linear(config.d_ff,    config.d_model, bias=config.bias), nn.Dropout(config.dropout))
    def forward(self, x): return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.d_model)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.d_model)
        self.ffn  = FeedForward(config)
    def forward(self, x, return_attention=False):
        if return_attention:
            a, w = self.attn(self.ln_1(x), return_attention=True)
            return x + a + self.ffn(self.ln_2(x + a)), w
        return x + self.attn(self.ln_1(x)) + self.ffn(self.ln_2(x + self.attn(self.ln_1(x))))

class DecoderOnlyLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config  = config
        self.embedding = InputEmbeddings(config)
        self.blocks    = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layers)])
        self.ln_f      = nn.LayerNorm(config.d_model)
        self.lm_head   = nn.Linear(config.d_model, config.vocab_size, bias=config.bias)
        self.lm_head.weight = self.embedding.token_embedding.weight
    def forward(self, input_ids, return_attention=False):
        x = self.embedding(input_ids)
        for block in self.blocks:
            x = block(x)
        return self.lm_head(self.ln_f(x))


In [ ]:
import sentencepiece as spm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

sp = spm.SentencePieceProcessor()
sp.Load(TOKENIZER_PATH)

model = DecoderOnlyLM(LMConfig())
model.to(device)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
load_result = model.load_state_dict(checkpoint["model_state"], strict=True)
model.eval()
print("=== CHECKPOINT VERIFICATION ===")
print(f"Checkpoint step: {checkpoint.get('step', 'Unknown')}")
print(f"Missing keys: {len(load_result.missing_keys)}")
print(f"Unexpected keys: {len(load_result.unexpected_keys)}")
print(f"Architecture matches: True (loaded successfully)")
print(f"Tokenizer vocab size: {sp.GetPieceSize()}")
print(f"Device: {device}")
print("===============================
")


In [ ]:
import re, json

# UNK surface form produced by SentencePiece for out-of-vocabulary tokens
_UNK_SURFACE = "\u2047"  # ⁇

def extract_answer(generated_text):
    """
    Deterministic first-word extraction — identical to evaluate_reasoning.py.
    1. Strip leading whitespace.
    2. Take the first contiguous non-whitespace/non-punctuation block.
    3. If the extracted token equals the UNK surface '⁇', return ''  (invalid).
    4. Also return boundary/diagnostic info for the notebook report.
    """
    text = generated_text.lstrip()
    match = re.match(r"^([^\s.,;!?\u0964]+)([\s.,;!?\u0964]*)(.*)$", text, flags=re.DOTALL)
    if match:
        extracted = match.group(1)
        boundary = match.group(2)
        tail = match.group(3)
        # Treat UNK surface as invalid
        if extracted == _UNK_SURFACE:
            return "", "UNK token (⁇) at boundary position — invalid", False, False, bool(boundary) or bool(tail)
        has_punct = any(c in boundary for c in ".,;!?\u0964")
        has_space = any(c in boundary for c in " \n\t")
        has_additional = bool(boundary) or bool(tail)
        reason = f"Encountered boundary: {repr(boundary[0])}" if boundary else "Reached end of string"
        return extracted, reason, has_punct, has_space, has_additional
    return "", "Empty or starts with boundary", False, False, False

torch.manual_seed(SEED)

lines = []
with open(TEST_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip(): lines.append(line.strip())

exact_matches = 0
prefix_matches = 0
eos_count = 0
max_len_count = 0
empty_count = 0
invalid_count = 0  # extracted == "" due to UNK or empty continuation
correct_first_word_but_extra_text = 0
correct_first_word_no_eos = 0

print("="*80)
print("RUNNING DIAGNOSTIC")
print("="*80)

for i in range(min(NUM_EXAMPLES, len(lines))):
    d = json.loads(lines[i])
    task_type = d.get("task_type", "")
    attribute = d.get("attribute", "")
    category = d.get("question_category", "")
    template_id = d.get("template_id", "")
    gt = str(d["answer"]).strip()
    
    prompt = f"प्रश्न: {d['premise']} {d['question']} उत्तर: "
    
    ids = torch.tensor([sp.Encode(prompt, out_type=int)], dtype=torch.long, device=device)
    gen_ids = []
    with torch.no_grad():
        for _ in range(MAX_NEW_TOKENS):
            if ids.shape[1] > 512: ids = ids[:, -512:]
            nxt = model(ids)[0,-1,:].argmax().item()
            if nxt == sp.eos_id(): break
            gen_ids.append(nxt)
            ids = torch.cat([ids, torch.tensor([[nxt]], device=device)], dim=1)
            
    raw_cont = sp.Decode(gen_ids)
    reached_eos = (len(gen_ids) < MAX_NEW_TOKENS)
    num_tokens = len(gen_ids)
    
    extracted, reason, has_punct, has_space, has_additional = extract_answer(raw_cont)
    is_exact = (extracted == gt)
    
    if is_exact: exact_matches += 1
    if raw_cont.lstrip().startswith(gt): prefix_matches += 1
    if reached_eos: eos_count += 1
    if num_tokens == MAX_NEW_TOKENS: max_len_count += 1
    if num_tokens == 0: empty_count += 1
    if not extracted: invalid_count += 1  # UNK or empty
    
    if is_exact and has_additional: correct_first_word_but_extra_text += 1
    if is_exact and not reached_eos: correct_first_word_no_eos += 1
    
    print(f"\n--- Example {i+1} ---")
    print(f"1. Example index      : {i+1}")
    print(f"2. Task type          : {task_type}")
    print(f"3. Attribute          : {attribute}")
    print(f"4. Question category  : {category}")
    print(f"5. Template family    : {template_id}")
    print(f"6. Ground-truth answer: {gt!r}")
    print(f"7. Exact prompt       : {prompt!r}")
    print(f"8. Raw continuation   : {raw_cont!r}")
    print(f"9. EOS reached?       : {reached_eos}")
    print(f"10. Generated tokens  : {num_tokens}")
    print(f"11. Extracted answer  : {extracted!r}")
    print(f"12. Exact-match result: {is_exact}")
    print(f"13. Stop reason       : {reason}")
    print(f"  -> Begins with GT?          {raw_cont.lstrip().startswith(gt)}")
    print(f"  -> Begins with incorrect?   {not raw_cont.lstrip().startswith(gt) and bool(extracted)}")
    print(f"  -> Punctuation immediately? {has_punct}")
    print(f"  -> Whitespace immediately?  {has_space}")
    print(f"  -> Contains extra text?     {has_additional}")

print("\n" + "="*80)
print("AGGREGATE DIAGNOSTICS")
print("="*80)
print(f"exact_match_accuracy      : {exact_matches}/{NUM_EXAMPLES} ({exact_matches/NUM_EXAMPLES*100:.1f}%)")
print(f"first-word/prefix (diag)  : {prefix_matches}/{NUM_EXAMPLES} ({prefix_matches/NUM_EXAMPLES*100:.1f}%) -- DIAGNOSTIC ONLY")
print(f"eos_rate                  : {eos_count/NUM_EXAMPLES*100:.1f}%")
print(f"max_length_rate           : {max_len_count/NUM_EXAMPLES*100:.1f}%")
print(f"empty_generation_rate     : {empty_count/NUM_EXAMPLES*100:.1f}%")
print(f"invalid_prediction_rate   : {invalid_count/NUM_EXAMPLES*100:.1f}%  (UNK or empty extracted)")
print(f"correct + extra text      : {correct_first_word_but_extra_text}/{NUM_EXAMPLES}")
print(f"correct + no EOS          : {correct_first_word_no_eos}/{NUM_EXAMPLES}")
